In [1]:
import pandas as pd
import os
config = pd.read_csv('config.csv')
repos_dir = config['repo_dir'][0]
import numpy as np
%matplotlib widget
import matplotlib.pyplot as plt
import plotly # built with plotly version '4.4.1'
import plotly.graph_objs as go
from plotly.offline import download_plotlyjs, init_notebook_mode, plot, iplot
from plotly.subplots import make_subplots
import sys
sys.path.append(os.path.join(repos_dir, 'sleep_research_io'))
from sleep_research_functions import load_sleep_data, get_metadata
import gc
# from tqdm import tqdm
import matplotlib.pyplot as plt
from scipy.signal import find_peaks
import seaborn as sns
from matplotlib import cm
import warnings
warnings.filterwarnings("ignore")

from sklearn.cluster import KMeans

# X = data[['accx_1sec','accy_1sec','accz_1sec']].dropna().values

from airgo_features import *

Entropy package not installed. Use complexity_features=0 when computing airgo features.


In [2]:

# def acceleration_position(data):
#     data['position_cluster'] = -1
# #     cluster_names = ['right_lateral','left_lateral','suspine','prone']
#     cluster_centers = np.array([
#         [-0.10528739, -0.90177368,  0.07899741],
#         [-0.14248938,  0.96973593, -0.06413331],
#         [-0.16258391, -0.05833847, -0.9190312 ],
#         [ 0.01891357,  0.09631911,  0.98882481]
#     ])

#     kmeans = KMeans(n_clusters=4, init=cluster_centers)
#     kmeans.cluster_centers_ = cluster_centers
#     index_valid = data[['accx_1sec','accy_1sec','accz_1sec']].dropna().index
#     data.loc[index_valid, 'position_cluster'] = kmeans.predict(data[['accx_1sec','accy_1sec','accz_1sec']].dropna())
    
#     return data

# def airgo_breath_peak_detection(data, fs=10):

#     data['band_detrended'] = data['band'].values - data['band'].reset_index()['band'].rolling(120*fs, center=True).mean().values
#     if 'movavg_0_5s' not in data.columns:
#         data['movavg_0_5s'] = data['band'].reset_index()['band'].rolling(int(0.5*fs), center=True).mean().values

#     data['movavg_0_5s_detrended'] = data['movavg_0_5s'].values - data['movavg_0_5s'].reset_index()['movavg_0_5s'].rolling(120*fs, center=True).mean().values
#     peaks, peak_prop = find_peaks(data.movavg_0_5s_detrended.values, prominence=1, distance=int(fs*1.2), width=int(0.4*fs), rel_height=1)
#     # rel_height=1 makes width be measured at bottom of peak candidates.
#     data['peaks'] = 0
#     data['peaks'].iloc[peaks] = 1
#     data['peak_prom']=0
#     data['peak_prom'].iloc[peaks] = peak_prop['prominences']

#     # CRITERIA 1: peak prominence must exceed an adaptive threshold, based on moving standard deviation:
#     data['movstd_1min'] = data['band_detrended'].reset_index()['band_detrended'].rolling(60*fs, center=True).std().values
#     data['movstd_30sec'] = data['band_detrended'].reset_index()['band_detrended'].rolling(30*fs, center=True).std().values
#     data['min_std_baseline'] = np.min(data[['movstd_30sec','movstd_1min']].values, axis=1)
#     data['peaks'][(data['peak_prom']<data['min_std_baseline']*0.65) & (data['peak_prom']>0) ] = 0.5
#     data.drop('min_std_baseline', axis=1, inplace=True)
    
#     # CRITERIA 2: further small and short breaths, compared to neighboring breaths, are removed.

#     num_corrected_small_short = 0
#     peaks = np.where(data['peaks']==1)[0]
#     prom = data['peak_prom'].iloc[peaks].values
#     data['diff_between_peaks'] = 0
#     data['ratio_time_between_peaks'] = 0
#     data['ratio_prom_pre'] = 0
#     data['ratio_prom_post'] = 0

#     for i in range(500):
#         diff_between_peaks = np.concatenate([[4*fs],np.diff(peaks)]) # add 40 samples = 4 sec as standard peak to peak value in the beinning.
#         ratio_time_between_peaks = np.concatenate([[1],np.divide(diff_between_peaks[1:], diff_between_peaks[:-1])])
#         ratio_prom_pre = np.divide(prom[1:], prom[:-1])
#         ratio_prom_pre = np.concatenate([[1], ratio_prom_pre])
#         ratio_prom_post = np.divide(prom[:-1], prom[1:])
#         ratio_prom_post = np.concatenate([ratio_prom_post, [1]])
#         small_and_short_breath = (ratio_prom_pre < 0.55) & (ratio_time_between_peaks<0.55) & (diff_between_peaks<fs*2)
#         small_and_short_breath.shape
#         idx_tmp = data['peaks'][data['peaks']==1][small_and_short_breath].index
#         data.loc[idx_tmp,'peaks'] = 0.6
#         num_corrected_previous = num_corrected_small_short
#         num_corrected_small_short = sum(data.peaks==0.6)

#         data['diff_between_peaks'].iloc[peaks] = diff_between_peaks
#         data['ratio_prom_pre'].iloc[peaks] = ratio_prom_pre
#         data['ratio_prom_post'].iloc[peaks] = ratio_prom_post
#         data['ratio_time_between_peaks'].iloc[peaks] = ratio_time_between_peaks

#         peaks = np.where(data['peaks']==1)[0]
#         prom = data['peak_prom'].iloc[peaks].values

#         if num_corrected_small_short == num_corrected_previous: break 
            
#     data.drop('diff_between_peaks', axis=1, inplace=True)
#     data.drop('ratio_prom_pre', axis=1, inplace=True)
#     data.drop('ratio_prom_post', axis=1, inplace=True)
#     data.drop('ratio_time_between_peaks', axis=1, inplace=True)
#     data.drop('movavg_0_5s_detrended', axis=1, inplace=True)
#     data.drop('band_detrended', axis=1, inplace=True)
#     data.drop('peak_prom', axis=1, inplace=True)


    
#     return data
            
    

In [3]:
# # ventilation
# def airgo_features(data, fs=10):
    
# #     data['movavg_1_2s'] = data['band'].rolling('1200ms').mean()
#     data['movavg_1_2s'] = data['band'].reset_index()['band'].rolling(int(1.2*fs), center=True).mean().values
#     data['movavg_0_5s'] = data['band'].reset_index()['band'].rolling(int(0.5*fs), center=True).mean().values
#     data['deriv1'] = np.concatenate([[0], np.diff(data['movavg_0_5s'])])
#     data['ventilation0']      = np.maximum(np.zeros(data['deriv1'].shape), data['deriv1'].values)
#     data['ventilation_3s']    = data['ventilation0'].rolling('3s').sum()*20
#     data['ventilation_8s']    = data['ventilation0'].rolling('8s').sum()*60/8
#     data['ventilation_10s']   = data['ventilation0'].rolling('10s').sum()*6
#     data['ventilation_12s']    = data['ventilation0'].rolling('12s').sum()*60/12
#     data['ventilation_15s']    = data['ventilation0'].rolling('15s').sum()*60/15
#     data['ventilation_30s']    = data['ventilation0'].rolling('30s').sum()*60/30
#     data['ventilation_60s']   = data['ventilation0'].rolling('60s').sum()
#     data['ventilation_5min']    = data['ventilation0'].rolling('5min').sum()/5
#     data['ventilation_10min']    = data['ventilation0'].rolling('10min').sum()/10
    
#     data['ventilation_10s_smooth'] = data['ventilation_10s'].rolling('10s').mean()
    
#     # PEAKS

#     data = airgo_breath_peak_detection(data)

#     ## 
#     data['rr_10s'] = data['peaks'].rolling(window='10s').sum() * 6
#     data['rr_60s'] = data['peaks'].rolling(window='60s').sum()
#     data['rr_10s_smooth'] = data['rr_10s'].rolling('10s').mean()

#     data['movmedian_1min'] = data['band'].reset_index()['band'].rolling(60*fs, center=True).quantile(0.5).values
#     data['movmedian_30sec'] = data['band'].reset_index()['band'].rolling(30*fs, center=True).quantile(0.5).values

#     data['IBI'] = np.nan
#     BB_interval = np.diff(np.where(data.peaks==1)[0])
#     BB_interval = np.concatenate([BB_interval[:1],BB_interval])
#     data['IBI'][data.peaks==1] = BB_interval/fs
#     data['IBI'][data['IBI']>15]=15
#     data['IBI'].interpolate(method='linear', limit=20*fs, limit_area='inside', inplace=True)

#     data['IBI_mean_5min'] = np.nan
#     data['IBI_mean_5min'][data.peaks==1] = data['IBI'][data.peaks==1].rolling('5min').mean()
#     data['IBI_std_5min'] = np.nan
#     data['IBI_std_5min'][data.peaks==1] = data['IBI'][data.peaks==1].rolling('5min').std()

#     data['ventilation_5min_deriv'] = data.ventilation_5min.diff().rolling('5min').mean()

#     # troughs, inhalation, exhalation time features:
#     data['troughs'] = 0
#     data['inht'] = np.nan
#     data['exht'] = np.nan
#     data['inht_exht_ratio'] = np.nan
#     data['inht_cycle_ratio'] = np.nan

#     loc_peak = data[data.peaks==1].index
#     trough_loc = [data.movavg_0_5s[loc_peak[iP]:loc_peak[iP+1]].argmin() for iP in range(loc_peak.shape[0]-1)]
#     data.loc[trough_loc, 'troughs'] = 1

#     idx_peak = np.where(data.peaks==1)[0]
#     idx_troughs = np.where(data.troughs==1)[0]

#     data.loc[loc_peak[1:], 'inht'] = (idx_peak[1:]-idx_troughs)/fs
#     data.loc[loc_peak[:-1], 'exht'] = (idx_troughs - idx_peak[:-1])/fs
#     data.loc[loc_peak, 'inht_exht_ratio'] = data['inht']/data['exht']
#     data['inht_exht_ratio_10sec'] = data['inht_exht_ratio'].reset_index()['inht_exht_ratio'].rolling(10*fs, center=True,min_periods=1).apply(lambda x: np.nanmean(x), raw=True).values
#     data.loc[loc_peak, 'inht_cycle_ratio'] = data['inht']/(data['exht']+data['inht'])
#     data['inht_cycle_ratio_10sec'] = data['inht_cycle_ratio'].reset_index()['inht_cycle_ratio'].rolling(10*fs, center=True,min_periods=1).apply(lambda x: np.nanmean(x), raw=True).values

#     # Tidal Volume per Breath features:
    
#     peaks_loc = data[data.peaks==1].index[1:]
#     troughs_loc = data[data.troughs==1].index
#     data['TVpB'] = np.nan
#     TVpB = np.array([data.movavg_0_5s[peak_loc] - data.movavg_0_5s[trough_loc] for peak_loc, trough_loc in zip(peaks_loc, troughs_loc)])
#     data.loc[peaks_loc, 'TVpB'] = TVpB
#     data['TVpB_regularity'] = np.nan
#     data['TVpB_regularity'].loc[peaks_loc[:-1]] = 1 - np.array([min(TVpB[i],TVpB[i+1])/max(TVpB[i],TVpB[i+1]) for i in range(len(TVpB)-1)])
#     data['TVpB_regularity_10sec'] = data['TVpB_regularity'].reset_index()['TVpB_regularity'].rolling(10*fs, center=True,min_periods=1).apply(lambda x: np.nanmean(x), raw=True).values

#     # accelerometer features:
#     g = 9.81
#     for acc in ['accx','accy', 'accz']:
#         data[acc] /=g
#         data[acc+'_1sec'] = data[acc].reset_index()[acc].rolling(fs, center=True).mean().values
#         data[acc+'_var_1sec'] = data[acc].reset_index()[acc].rolling(fs, center=True).var().values
#         data[acc+'_var_10sec'] = data[acc].reset_index()[acc].rolling(10*fs, center=True).var().values

#     data['acc_ai_1sec'] = np.sqrt(np.maximum([0],np.mean([data['accx_var_1sec'], data['accy_var_1sec'], data['accz_var_1sec']], axis=0)))
#     data['acc_ai_10sec'] = np.sqrt(np.maximum([0],np.mean([data['accx_var_10sec'], data['accy_var_10sec'], data['accz_var_10sec']], axis=0)))
#     data['acc_enmo'] = np.maximum([0],np.sqrt(data.accx**2+data.accy**2+data.accz**2)-0.98)
#     data['acc_enmo_1sec'] = data['acc_enmo'].reset_index()['acc_enmo'].rolling(fs, center=True).mean().values
#     data['acc_enmo_10sec'] = data['acc_enmo'].reset_index()['acc_enmo'].rolling(10*fs, center=True).mean().values
#     # get position cluster (assumption: device is worn on the chest; 4 clusters)
#     data = acceleration_position(data)
#     for acc in ['accx','accy', 'accz']:
#         data.drop(acc+'_var_1sec', axis=1, inplace=True)
#         data.drop(acc+'_var_10sec', axis=1, inplace=True)
    
#     return data


In [4]:
# loc_peak = data[data.peaks==1].index
# trough_loc = [data.movavg_0_5s[loc_peak[iP]:loc_peak[iP+1]].argmin() for iP in range(loc_peak.shape[0]-1)]

In [5]:
# s_microlist = pd.read_csv('sleeplab_micro_list.csv')
# i_microlist = pd.read_csv('icu_micro_list.csv')
# i_microlist.head()
# sleeplab_files = ['psg_airgo_10hz_'+str(x).zfill(3)+'.h5' for x in s_microlist.study_id]
# icu_files = ['icusleep_'+str(x)+'.h5' for x in i_microlist.night_id]
# i_microlist['comment'] = np.nan
# s_microlist['file'] = sleeplab_files
# i_microlist['file'] = icu_files
# print(i_microlist.head(2))
# print(s_microlist.head(2))

In [6]:
# files_directory = '/media/ssd2/ICU_sleep_data/biosignals_10Hz_data_daynight/'
files_directory ='.'
# all_files = os.listdir(files_directory)
# all_files.sort()

# for filename in tqdm(all_files):

sleeplab_files = ['psg_airgo_10hz_001.h5', 'psg_airgo_10hz_002.h5', 'psg_airgo_10hz_003.h5', 'psg_airgo_10hz_004.h5', 'psg_airgo_10hz_005.h5',]
icu_files = ['icusleep_021_n02.h5', 'icusleep_025_n04.h5', 'icusleep_079_n05.h5']
# sleeplab_files = ['psg_airgo_10hz_370.h5']
# icu_files = []

filename = sleeplab_files[0]
dall = {}
statistics = pd.DataFrame()

for filename in (sleeplab_files+icu_files)[-1:]:
    try:
        filepath = os.path.join(files_directory, filename)
        signals_contained_tmp, hdr = get_metadata(filepath)
    except:
        continue
    data, hdr = load_sleep_data(filepath, idx_to_datetime=1)
    data.columns = [x.lower() for x in data.columns]
    
    if 'psg' in filename:
        data = data[['band','accx','accy','accz','stage','apnea']]
        ptype = 'sleeplab'
#         comment = s_microlist[s_microlist.file==filename]['comment'].values[0]
    else: 
        data = data[['band','accx','accy','accz']]
        ptype = 'icu'
#         comment = i_microlist[i_microlist.file==filename]['comment'].values[0]
        
    fs = hdr['fs']
    print(f'fs: {fs}')

    data = compute_airgo_features(data, complexity_features=0)
#     signals_to_plot = ['band', 'movmedian_1min', 'rr_10s', 'rr_10s_smooth', 
#                        'ventilation_10s_smooth', 'ventilation_10s',
#                        'IBI','peaks', 'movavg_0_5s']
#     if 'psg' in filename: signals_to_plot += ['stage', 'apnea']
#     fig = icu_sleep_plot(data[signals_to_plot])
#     fig.update_layout(title=filename)
#     plot(fig, filename=filename+'.html', auto_open=True)

    dall[filename] = data
    df = pd.DataFrame()
    df['filename'] = [filename]
    df['ptype'] = [ptype]
#     df['comment'] = [comment]
    df['IBI_mean'] = [data['IBI'][data.peaks==1].mean()]
    df['IBI_std'] = [data['IBI'][data.peaks==1].std()]
    df['IBI_median'] = [data['IBI'][data.peaks==1].median()]
    df['IBI_Q75'] = [data['IBI'][data.peaks==1].quantile(0.75)]
    df['IBI_Q25'] = [data['IBI'][data.peaks==1].quantile(0.25)]
    df['IBI_RMSSD'] = [np.sqrt(np.mean(data['IBI'][data.peaks==1].diff().values[2:]**2))]
    df['IBI_SDA'] = [data['IBI_mean_5min'].std()]
    df['IBI_ASD'] = [data['IBI_std_5min'].mean()]

    statistics = pd.concat([statistics, df])


fs: 10


In [9]:
statistics

,filename,ptype,IBI_mean,IBI_std,IBI_median,IBI_Q75,IBI_Q25,IBI_RMSSD,IBI_SDA,IBI_ASD
0,icusleep_079_n05.h5,icu,3.069206,0.689685,3.1,3.4,2.7,0.738438,0.374757,0.513584


In [76]:
# from statsmodels.regression.rolling import RollingOLS

# data['ventilation_5min_deriv'] = data.ventilation_5min.diff().dropna().rolling('5min').apply(lambda x: np.polyfit(range(len(pd.Series(x).dropna())), pd.Series(x).dropna(), 1)[0], raw=True)
# data['ventilation_5min_deriv2'] = (10*data.ventilation_5min).diff().rolling('5min', min_periods=1).apply(lambda x: np.polyfit(range(len(x)), x, 1)[0])

In [155]:
lw = 0.8
plt.figure(figsize=(12,8))
ax = plt.subplot(511)
plt.plot(data.band,c='dodgerblue', alpha=0.7, lw=lw)
plt.plot(data.movavg_0_5s,c='blue', lw=lw)
# plt.scatter(data.index[data.peaks==1], data.movavg_0_5s[data.peaks==1],c='r',s=8, zorder=3)
# plt.scatter(data.index[data.troughs==1], data.movavg_0_5s[data.troughs==1],c='g',s=11, zorder=3)

ax2 =plt.subplot(512, sharex=ax)
# plt.plot(data.index[data.peaks==1], data.inht[data.peaks==1],c='blue')
# plt.plot(data.index[data.peaks==1], data.exht[data.peaks==1],c='red')
# plt.plot(data.index[data.peaks==1], data.inht_exht_ratio[data.peaks==1],c='green')
plt.plot(data.index[data.peaks==1], data.inht_cycle_ratio[data.peaks==1],c=(0.1,0.8,0.1), alpha=0.6, lw=lw)
plt.plot(data['inht_cycle_ratio_10sec'],c='green', lw=lw)
plt.ylabel('Inhalation Time (%)')
ax3 =plt.subplot(513, sharex=ax)
# plt.plot(data.index[data.peaks==1], data.inht_exht_ratio[data.peaks==1],c=(0.1,0.1,0.1), alpha=0.6, lw=lw)
# plt.plot(data['inht_exht_ratio_10sec'],c='black', lw=lw)
# plt.ylabel('In/Exhalation Time')
plt.plot(data.ventilation_15s, c=(0.6,0.3,0.3), alpha=0.6, lw=lw)
plt.plot(data.ventilation_5min, c=(1,0,0), alpha=1, lw=lw)
plt.ylabel('Tidal Volume')
ax4 =plt.subplot(514, sharex=ax)
plt.plot(data.index[data.peaks==1], data.TVpB[data.peaks==1],c=(0.5,0.5,0.5), alpha=0.6, lw=lw)
ax4 =plt.subplot(515, sharex=ax)
plt.plot(data.index[data.peaks==1][1:-1], TVpB_regularity,c=(0.5,0.5,0.5), alpha=0.6, lw=lw)
plt.plot(data.index[data.peaks==1][1:-1], pd.Series(TVpB_regularity).rolling(10).mean(),c=(0,0.3,0.3), alpha=0.6, lw=lw)



# plt.plot(data.ventilation_5min.diff().rolling('5min').mean(), c=(1,0.5,0.5), alpha=1, lw=lw)

# plt.plot(data.ventilation_5min.diff().rolling('1min').mean(), c=(0.5,0.6,0.5), alpha=1, lw=lw)

# plt.plot(data.index, np.zeros(len(data),))
# plt.plot(data.ventilation_5min.diff().rolling('1min').mean().cumsum())
# plt.plot(data.ventilation_5min.diff().diff().rolling('10s').mean()-2, c=(0.8,0.2,0.2), alpha=1, lw=lw)



Canvas(toolbar=Toolbar(toolitems=[('Home', 'Reset original view', 'home', 'home'), ('Back', 'Back to previous …

In [107]:
from scipy.stats import zscore
plt.figure()
plt.subplot(211)
deriv_values = data['ventilation_5min_deriv'].dropna()[10*60*fs:]
print(np.std(deriv_values))
deriv_values = deriv_values/np.std(deriv_values)
plt.hist(deriv_values)


Canvas(toolbar=Toolbar(toolitems=[('Home', 'Reset original view', 'home', 'home'), ('Back', 'Back to previous …

0.11921529376619155


(array([  1318.,   1720.,   4689.,   7187.,  87685., 182451., 109772.,
         26523.,   3196.,   1459.]),
 array([-5.44292582, -4.47112656, -3.49932731, -2.52752806, -1.55572881,
        -0.58392955,  0.3878697 ,  1.35966895,  2.3314682 ,  3.30326745,
         4.27506671]),
 <a list of 10 Patch objects>)

In [10]:

import pandas as pd
import numpy as np
#matplotlib widget
import plotly # built with plotly version '4.4.1'
import plotly.graph_objs as go
from plotly.offline import download_plotlyjs, init_notebook_mode, plot, iplot
from plotly.subplots import make_subplots
import sys
import matplotlib.pyplot as plt

def airgo_multiple_lines_plot(data):

    band = (data.band.values-np.median(data.band.values))/(data.band.quantile(0.75)-data.band.quantile(0.25))
    min_per_row = 15
    minutes_data = data.shape[0]/10/60
    
    # define the ids each row
    nrow = int(minutes_data/min_per_row)+1
    row_ids = np.array_split(np.arange(len(data)),nrow)
    row_ids.reverse()

    fig = plt.figure(figsize=(24, int(nrow*1.5)))
    ax = fig.add_subplot(111)
    row_height = 2

    for ri in range(nrow):
        ax.plot(band[row_ids[ri]] + ri * row_height, c='k', lw=0.5) # data.index[row_ids[ri]]
        if (nrow-ri)%2==0:
            ax.text(-300, ri * row_height-1, str(data.index[row_ids[ri]][0].time())[:8], rotation=90)
        if (nrow-ri)%5==0:
            ax.text(-450, ri * row_height-1, str(data.index[row_ids[ri]][0].date()), rotation=90)

#     ax.set_xlim([0, max([len(x) for x in row_ids])])
    ax.axis('off')
    plt.tight_layout()
       
    return fig


def covid_plot(data, trace_linewidth=1, labelfontsize=10, legend_position = [0.2, 1.15], legend_font_size=10, annotations=[]):
    '''
    Plot designed for the 10Hz data, plots AirGo, Blood Pressure, Heart Rate, and SpO2.
    Input: sleep research-formatted data
    Output: fig (plotly figure instance)
    '''
    
    airgo_available = 'band' in data.columns
    BP_available = any(['ART1D' in data.columns, 'ART1S' in data.columns, 'ART2D' in data.columns, 'ART2S' in data.columns, 'NBPD' in data.columns, 'NBPS' in data.columns])
    HR_available = 'HR' in data.columns
    SPO2_available = 'SPO2%' in data.columns
    RR_available = any(['RESP' in data.columns, 'Vent Rate' in data.columns])
    MV_available = any(['MV' in data.columns, 'SPONTMV' in data.columns])
    PTM_available = 'INSPTM' in data.columns

    num_traces = max(1,sum([airgo_available, BP_available, HR_available, SPO2_available, RR_available, MV_available, PTM_available]))

    fig = make_subplots(rows=num_traces, cols=1, shared_xaxes=True, shared_yaxes=False, specs=[[{"secondary_y": False}]] * num_traces )

    traces = []
    i_trace=1

    if airgo_available:
        trace_AirGo = go.Scatter(x=data.index, y=data.band, name = 'AirGo', hoverinfo = 'x+y', line = dict(color = 'dodgerblue', width = trace_linewidth), opacity = 0.8 )
        fig.add_trace(trace_AirGo, i_trace, 1)
        fig.update_yaxes(title_text="Circumference <br> Thorax (a.u.)", row=i_trace, col=1, range=[data['band'].quantile(0.01),data['band'].quantile(0.99)], title_font=dict(size=labelfontsize))
        i_trace+=1

    if 'RESP' in data.columns:
        trace_RR = go.Scatter(x=data.index, y=data.RESP, name = 'Resp.Rate', hoverinfo = 'x+y', line = dict(color = 'dodgerblue', width = trace_linewidth), opacity = 0.8)
        fig.add_trace(trace_RR, i_trace, 1)
    if 'Vent Rate' in data.columns:
        trace_VR = go.Scatter(x=data.index, y=data['Vent Rate'], name = 'Vent.Rate', hoverinfo = 'x+y', line = dict(color = 'dodgerblue', width = trace_linewidth), opacity = 0.8)
        fig.add_trace(trace_VR, i_trace, 1)
    if RR_available: 
        fig.update_yaxes(title_text="Resp. Rate", row=i_trace, col=1, title_font=dict(size=labelfontsize))
        if len(annotations)>0:
            for annotation in annotations:
                trace_annotation = go.Scatter(x=[annotation[0], annotation[0], annotation[1], annotation[1], annotation[0]],
                                              y=[data.RESP.quantile(0.005), data.RESP.quantile(0.995), data.RESP.quantile(0.995), data.RESP.quantile(0.005), data.RESP.quantile(0.005)],
                                              hovertemplate = annotation[2], name=annotation[2], #   hoverinfo = annotation[2], 
                                              line = dict(color = 'green', width = trace_linewidth), opacity=0.5,  fill="toself", showlegend=False)
                fig.add_trace(trace_annotation, i_trace, 1)
        i_trace+=1

    if 'MV' in data.columns:
        trace_MV = go.Scatter(x=data.index, y=data.MV, name = 'Minute Ventilation', hoverinfo = 'x+y', line = dict(color = 'dodgerblue', width = trace_linewidth), opacity = 0.8)
        fig.add_trace(trace_MV, i_trace, 1)
    if 'SPONTMV' in data.columns:
        trace_SMV = go.Scatter(x=data.index, y=data['SPONTMV'], name ='Spont. MV', hoverinfo = 'x+y', line = dict(color = 'dodgerblue', width = trace_linewidth,  dash = 'dashdot'), opacity = 0.8)
        fig.add_trace(trace_SMV, i_trace, 1)
    if MV_available: 
        fig.update_yaxes(title_text="Minute Ventilation", row=i_trace, col=1, title_font=dict(size=labelfontsize))
        i_trace+=1 
         
    if PTM_available:
        trace_PTM = go.Scatter(x=data.index, y=data.INSPTM, name = 'Ins. Transmural Pressure', hoverinfo = 'x+y', line = dict(color = 'green', width = trace_linewidth), opacity = 0.8)
        fig.add_trace(trace_PTM, i_trace, 1) 
        fig.update_yaxes(title_text="Ins. Transmural Pressure", row=i_trace, col=1, title_font=dict(size=labelfontsize))

        i_trace+=1 
        
    if SPO2_available:
        trace_SPO2 = go.Scatter(x=data.index, y=data['SPO2%'], name = 'SpO2%', hoverinfo = 'x+y', line = dict(color = 'navy', width = trace_linewidth), opacity = 0.8)
        fig.add_trace(trace_SPO2, i_trace, 1)
        fig.update_yaxes(title_text="SpO2 (%)", row=i_trace, col=1, range=[data['SPO2%'].quantile(0.01),data['SPO2%'].quantile(1)+0.5], title_font=dict(size=labelfontsize))
        i_trace+=1

    if HR_available:
        trace_HR = go.Scatter(x=data.index, y=data.HR, name = 'Heart Rate', hoverinfo = 'x+y', line = dict(color = 'crimson', width = trace_linewidth), opacity = 0.8)
        fig.add_trace(trace_HR, i_trace, 1)
        fig.update_yaxes(title_text="HR(bpm)", row=i_trace, col=1, range=[data['HR'].quantile(0.005),data['HR'].quantile(0.995)], title_font=dict(size=labelfontsize))
        i_trace+=1


    showlegend_ART2D = True
    showlegend_ART2S = True

    if 'ART1D' in data.columns:
        trace_ART1D = go.Scatter(x=data.index, y=data.ART1D, name = 'Art. BP diastolic', hoverinfo = 'x+y', line = dict(color = 'magenta', width = trace_linewidth), opacity = 0.8)
        fig.add_trace(trace_ART1D, i_trace, 1)
#         art1d_data_len = len(data.ART1D.dropna())
#     art1s_data_len = len(data.ART1S.dropna())
        showlegend_ART2D = False
    if 'ART1S' in data.columns:
        trace_ART1S = go.Scatter(x=data.index, y=data.ART1S, name = 'Art. BP systolic', hoverinfo = 'x+y', line = dict(color = 'orange', width = trace_linewidth), opacity = 0.8)
        fig.add_trace(trace_ART1S, i_trace, 1)
        showlegend_ART2S = False
    if 'ART2D' in data.columns:
        trace_ART2D = go.Scatter(x=data.index, y=data.ART2D, name = 'Art. BP diastolic 2', hoverinfo = 'x+y', line = dict(color = 'magenta', width = trace_linewidth), opacity = 0.8, showlegend=False)
        fig.add_trace(trace_ART2D, i_trace, 1)
    if 'ART2S' in data.columns:
        trace_ART2S = go.Scatter(x=data.index, y=data.ART2S, name = 'Art. BP systolic 2', hoverinfo = 'x+y', line = dict(color = 'orange', width = trace_linewidth), opacity = 0.8, showlegend=False)
        fig.add_trace(trace_ART2S, i_trace, 1)
    if 'NBPD' in data.columns:
        trace_NBPD = go.Scatter(x=data.index, y=data.NBPD, name = 'NonInv. BP diastolic', hoverinfo = 'x+y', line = dict(color = 'magenta', width = trace_linewidth, dash = 'dashdot'), opacity = 0.8 )
        fig.add_trace(trace_NBPD, i_trace, 1)
    if 'NBPS' in data.columns:
        trace_NBPS = go.Scatter(x=data.index, y=data.NBPS, name = 'NonInv. BP systolic', hoverinfo = 'x+y', line = dict(color = 'orange', width = trace_linewidth, dash = 'dashdot'), opacity = 0.8 )
        fig.add_trace(trace_NBPS, i_trace, 1)
    if BP_available: 
        fig.update_yaxes(title_text="BP (mmHg)", row=i_trace, col=1, title_font=dict(size=labelfontsize))
        i_trace+=1


    fig.update_layout(plot_bgcolor='rgb(237,237,237)')     # 'white'
    fig.update_layout(legend=dict(x=legend_position[0], y=legend_position[1], orientation = 'h'), font=dict(size=legend_font_size, color="black"))

    return fig

In [75]:


# def icu_sleep_plot(data, trace_linewidth=1, labelfontsize=10, legend_position = [0.2, 1.15], legend_font_size=10):
#     '''
#     Plot designed for the 10Hz data, plots AirGo, Blood Pressure, Heart Rate, and SpO2.
#     Input: sleep research-formatted data
#     Output: fig (plotly figure instance)
#     '''
    
#     data.columns = [x.lower() for x in data.columns]
#     annotation_available = any(['stage' in data.columns, 'apnea' in data.columns])
#     airgo_available = any(['band' in data.columns, 'movavg_0_5s' in data.columns])
#     BP_available = any(['ART1D' in data.columns, 'ART1S' in data.columns, 'NBPD' in data.columns, 'NBPS' in data.columns])
#     HR_available = any(['HR' in data.columns, 'hr' in data.columns])
#     SPO2_available = any(['SPO2%' in data.columns, 'spo2%' in data.columns])
#     RR_available = any(['rr_10s' in data.columns, 'rr_10s_smooth' in data.columns])
#     ventilation_available = any(['ventilation_10s_smooth' in data.columns, 'ventilation_10s' in data.columns] )
#     ibi_available = any(['ibi' in data.columns, 'ibi_std_5min' in data.columns])
#     inht_exht_available = any(['inht_cycle_ratio_10sec' in data.columns])
#     airgo_actigraphy_available = any(['acc_ai_10s' in data.columns, 'accy_1s' in data.columns, 'position_cluster' in data.columns])
    
#     num_traces = sum([annotation_available, airgo_actigraphy_available, airgo_available, BP_available, HR_available, SPO2_available, ibi_available, RR_available, ventilation_available, inht_exht_available])

#     fig = make_subplots(rows=num_traces, cols=1, shared_xaxes=True, shared_yaxes=False, specs=[[{"secondary_y": False}]] * num_traces )

#     traces = []
#     i_trace=1

#     if annotation_available:
#         trace_Sleep = go.Scatter(x=data.index[::5], y=data.stage[::5], name = 'Stage', hoverinfo = 'x+y', line = dict(color = 'Crimson', width = 1), opacity = 0.8 )
#         fig.add_trace(trace_Sleep, i_trace, 1)
#         trace_Apnea = go.Scatter(x=data.index[::5], y=data.apnea[::5], name = 'Apnea', hoverinfo = 'x+y', line = dict(color = 'DeepSkyBlue', width = 1), opacity = 0.8)
#         fig.add_trace(trace_Apnea, i_trace, 1)
#         fig.update_yaxes(title_text="Stage, Apnea", row=i_trace, col=1, title_font=dict(size=labelfontsize), range=[-0.1,6.1], tickvals=[5,4,3,2,1], ticktext =['W','R','N1','N2','N3'], color = 'red')
#         i_trace+=1

#     if airgo_actigraphy_available:
#         offset_position = 0
#         if 'acc_ai_10sec' in data.columns:
#             trace = go.Scatter(x=data.index, y=data.acc_ai_10sec, name = 'Actigraphy Activity', hoverinfo = 'x+y', line = dict(color = 'peru', width = trace_linewidth), opacity = 0.8 )
#             fig.add_trace(trace, i_trace, 1)
#             range_signal = [-0.01, data['acc_ai_10sec'].quantile(0.99)]
#             offset_position = data['acc_ai_10sec'].quantile(0.99)+0.1
            
#         if 'position_cluster' in data.columns:
#             cluster_names = ['R.Lat.','L.Lat.','Suspine','Prone']
#             cluster_colors = ['red','green','blue','black']
#             for i in range(4):
#                 cluster_loc = data[data.position_cluster==i].index[::5]
#                 trace = go.Scatter(x=cluster_loc, y=offset_position*np.ones(len(cluster_loc),), name = 'Act. '+cluster_names[i], hoverinfo = 'x+y', line = dict(color = cluster_colors[i], width = trace_linewidth), opacity = 0.8 )
#                 fig.add_trace(trace, i_trace, 1)
            
#             range_signal = [-0.01, offset_position+0.02]

#         fig.update_yaxes(title_text="Actigraphy <br> Activity (g)", row=i_trace, col=1, range=range_signal, title_font=dict(size=labelfontsize))
#         i_trace+=1
        
#     if airgo_available:
#         airgo_colors = ['dodgerblue', 'blue']
#         i_airgo = 0
#         if 'band' in data.columns:
#             trace_AirGo = go.Scatter(x=data.index, y=data.band, name = 'AirGo', hoverinfo = 'x+y', line = dict(color = airgo_colors[i_airgo], width = trace_linewidth), opacity = 0.5 )
#             fig.add_trace(trace_AirGo, i_trace, 1)
#             i_airgo += 1
#             range_signal = [data['band'].quantile(0.01),data['band'].quantile(0.99)]
#         if 'movavg_0_5s' in data.columns:
#             trace_AirGo = go.Scatter(x=data.index, y=data.movavg_0_5s, name = 'AirGo', hoverinfo = 'x+y', line = dict(color = airgo_colors[i_airgo], width = trace_linewidth), opacity = 0.8 )
#             fig.add_trace(trace_AirGo, i_trace, 1)
#             range_signal = [data['movavg_0_5s'].quantile(0.01),data['movavg_0_5s'].quantile(0.99)]
#         if 'movmedian_1min' in data.columns:
#             trace_AirGo = go.Scatter(x=data.index, y=data['movmedian_1min'], name = 'AirGo 1-min median', hoverinfo = 'x+y', line = dict(color = 'black', width = trace_linewidth), opacity = 0.8 )
#             fig.add_trace(trace_AirGo, i_trace, 1)
#         if 'movavg_1min' in data.columns:
#             trace_AirGo = go.Scatter(x=data.index, y=data['movavg_1min'], name = 'AirGo 1-min mean', hoverinfo = 'x+y', line = dict(color = 'black', width = trace_linewidth), opacity = 0.8 )
#             fig.add_trace(trace_AirGo, i_trace, 1)
            
#         if 'peaks' in data.columns:
#             peaks = data['peaks']
#             trace_AirGo = go.Scatter(x=data.index[peaks==1], y=data.movavg_0_5s[peaks==1], 
#                         name = 'detected peak', hoverinfo = 'x+y', fillcolor='red', 
#                                      mode='markers', opacity = 0.8,
#                                      marker=dict(size=4, color='red')
#                                     )
#             fig.add_trace(trace_AirGo, i_trace, 1)
            
#         fig.update_yaxes(title_text="Circumference <br> Thorax (a.u.)", row=i_trace, col=1, range=range_signal, title_font=dict(size=labelfontsize))
#         i_trace+=1

#     if ventilation_available:
#         trace = go.Scatter(x=data.index, y=data.ventilation_10s_smooth, name = 'Minute Ventilation', hoverinfo = 'x+y', line = dict(color = 'purple', width = trace_linewidth), opacity = 0.8)
#         fig.add_trace(trace, i_trace, 1)
#         range_signal = [data['ventilation_10s_smooth'].quantile(0.01),data['ventilation_10s_smooth'].quantile(0.99)]
#         fig.update_yaxes(title_text="Minute Ventilation", row=i_trace, col=1, title_font=dict(size=labelfontsize), range=range_signal)
#         i_trace+=1
        
#     if RR_available:

#         trace_RR = go.Scatter(x=data.index, y=data.rr_10s_smooth, name = 'Resp.Rate', hoverinfo = 'x+y', line = dict(color = 'orangered', width = trace_linewidth), opacity = 0.8)
#         range_signal = [data['rr_10s_smooth'].quantile(0.01),data['rr_10s_smooth'].quantile(0.99)]
#         fig.add_trace(trace_RR, i_trace, 1)
#         fig.update_yaxes(title_text="Resp. Rate", row=i_trace, col=1, title_font=dict(size=labelfontsize), range=range_signal)
#         i_trace+=1
        
#     if ibi_available:
#         if 'ibi' in data.columns:
#             trace = go.Scatter(x=data.index, y=data.ibi, name = 'Inter-Breath Interval', hoverinfo = 'x+y', line = dict(color = 'lightsalmon', width = trace_linewidth), opacity = 0.8)
#             fig.add_trace(trace, i_trace, 1)
#         if 'ibi_std_5min' in data.columns:
#             trace = go.Scatter(x=data.index[np.logical_not(pd.isna(data['ibi_std_5min']))], y=data.ibi_std_5min[np.logical_not(pd.isna(data['ibi_std_5min']))], 
#                                name = 'Inter-Breath Interval STD', hoverinfo = 'x+y', line = dict(color = 'orangered', width = trace_linewidth), opacity = 0.8)
#             fig.add_trace(trace, i_trace, 1) 
            
#         fig.update_yaxes(title_text="IBI", row=i_trace, col=1, title_font=dict(size=labelfontsize))
#         i_trace+=1
        
#     if inht_exht_available:
#         if 'inht_cycle_ratio_10sec' in data.columns:
#             trace = go.Scatter(x=data.index, y=data.inht_cycle_ratio_10sec, name = 'Inhalation Time Ratio', hoverinfo = 'x+y', line = dict(color = 'green', width = trace_linewidth), opacity = 0.8)
#             fig.add_trace(trace, i_trace, 1)
#             fig.update_yaxes(title_text="Inhalation <br> Time (%)", row=i_trace, col=1, title_font=dict(size=labelfontsize))
#             i_trace+=1       
                              
                              
#     if 'ART1D' in data.columns:
#         trace_ART1D = go.Scatter(x=data.index, y=data.ART1D, name = 'Art. BP diastolic', hoverinfo = 'x+y', line = dict(color = 'magenta', width = trace_linewidth), opacity = 0.8)
#         fig.add_trace(trace_ART1D, i_trace, 1)
#     if 'ART1S' in data.columns:
#         trace_ART1S = go.Scatter(x=data.index, y=data.ART1S, name = 'Art. BP systolic', hoverinfo = 'x+y', line = dict(color = 'orange', width = trace_linewidth), opacity = 0.8)
#         fig.add_trace(trace_ART1S, i_trace, 1)
#     if 'NBPD' in data.columns:
#         trace_NBPD = go.Scatter(x=data.index, y=data.NBPD, name = 'NonInv. BP diastolic', hoverinfo = 'x+y', line = dict(color = 'magenta', width = trace_linewidth, dash = 'dashdot'), opacity = 0.8 )
#         fig.add_trace(trace_NBPD, i_trace, 1)
#     if 'NBPS' in data.columns:
#         trace_NBPS = go.Scatter(x=data.index, y=data.NBPS, name = 'NonInv. BP systolic', hoverinfo = 'x+y', line = dict(color = 'orange', width = trace_linewidth, dash = 'dashdot'), opacity = 0.8 )
#         fig.add_trace(trace_NBPS, i_trace, 1)
#     if BP_available: 
#         fig.update_yaxes(title_text="BP (mmHg)", row=i_trace, col=1, title_font=dict(size=labelfontsize))
#         i_trace+=1

#     if HR_available:
#         trace_HR = go.Scatter(x=data.index, y=data.hr, name = 'Heart Rate', hoverinfo = 'x+y', line = dict(color = 'crimson', width = trace_linewidth), opacity = 0.8)
#         fig.add_trace(trace_HR, i_trace, 1)
#         fig.update_yaxes(title_text="HR(bpm)", row=i_trace, col=1, range=[data['hr'].quantile(0.01),data['hr'].quantile(0.99)], title_font=dict(size=labelfontsize))
#         i_trace+=1

#     if SPO2_available:
#         trace_SPO2 = go.Scatter(x=data.index, y=data['spo2%'], name = 'SpO2%', hoverinfo = 'x+y', line = dict(color = 'navy', width = trace_linewidth), opacity = 0.8)
#         fig.add_trace(trace_SPO2, i_trace, 1)
#         fig.update_yaxes(title_text="SpO2 (%)", row=i_trace, col=1, range=[data['spo2%'].quantile(0.01),data['spo2%'].quantile(1)+0.5], title_font=dict(size=labelfontsize))
#         i_trace+=1

#     fig.update_layout(plot_bgcolor='rgb(237,237,237)')     # 'white'
#     fig.update_layout(legend=dict(x=legend_position[0], y=legend_position[1], orientation = 'h'), font=dict(size=legend_font_size, color="black"))

#     return fig

In [76]:
data_sel = data

In [77]:
signals_to_plot = ['acc_ai_10sec', 'position_cluster', 'movavg_0_5s', 'movavg_1min',  'rr_10s_smooth', 'IBI_std_5min', 'ventilation_10s_smooth', 'inht_cycle_ratio_10sec']

fig1 = icu_sleep_plot(data_sel[signals_to_plot].iloc[::2])
plot(fig1, filename = 'tmpcovidplot_downs.html')

'tmpcovidplot_downs.html'

In [29]:
data_sel['IBI_std_5min'].dropna()

2019-05-07 20:01:02.300    0.000000
2019-05-07 20:01:03.700    0.115470
2019-05-07 20:01:05.400    0.236291
2019-05-07 20:01:07.300    0.311448
2019-05-07 20:01:08.800    0.278687
                             ...   
2019-05-07 21:59:44.500    0.437018
2019-05-07 21:59:48.200    0.442102
2019-05-07 21:59:51.200    0.442102
2019-05-07 21:59:54.300    0.442052
2019-05-07 21:59:57.500    0.442366
Name: IBI_std_5min, Length: 2637, dtype: float64

In [17]:
data.columns

Index(['band', 'accx', 'accy', 'accz', 'movavg_0_5s', 'movavg_1_2s', 'deriv1',
       'ventilation0', 'ventilation_3s', 'ventilation_8s', 'ventilation_10s',
       'ventilation_12s', 'ventilation_15s', 'ventilation_30s',
       'ventilation_60s', 'ventilation_5min', 'ventilation_10min',
       'ventilation_10s_smooth', 'hypo_10_60', 'movmedian_5min',
       'movmedian_10min', 'movstd_8s', 'movstd_12s', 'movstd_15s',
       'movstd_10s', 'movstd_30s', 'movstd_60s', 'movstd_5min', 'movstd_10min',
       'peaks', 'movstd_1min', 'movstd_30sec', 'rr_10s', 'rr_60s',
       'rr_10s_smooth', 'movmedian_1min', 'movmedian_30sec', 'IBI',
       'IBI_mean_5min', 'IBI_std_5min', 'ventilation_5min_deriv', 'troughs',
       'inht', 'exht', 'inht_exht_ratio', 'inht_cycle_ratio',
       'inht_exht_ratio_10sec', 'inht_cycle_ratio_10sec', 'TVpB',
       'TVpB_regularity', 'TVpB_regularity_10sec', 'accx_1sec', 'accy_1sec',
       'accz_1sec', 'acc_ai_1sec', 'acc_ai_10sec', 'acc_enmo', 'acc_enmo_1sec',
   

In [103]:
deriv_values.kurtosis()

3.408939174296727

In [104]:
deriv_values.skew()

-0.5165932710199133

In [105]:
deriv_values.mean()

0.01409445073215654

In [106]:
deriv_values.std()

1.0000011737109888

In [92]:
data['ventilation_5min_deriv'].dropna()[5*60*fs:]

2019-05-07 20:05:00.100    0.673800
2019-05-07 20:05:00.200    0.675253
2019-05-07 20:05:00.300    0.676613
2019-05-07 20:05:00.400    0.677613
2019-05-07 20:05:00.500    0.677653
                             ...   
2019-05-08 07:59:59.600   -0.085547
2019-05-08 07:59:59.700   -0.085320
2019-05-08 07:59:59.800   -0.084787
2019-05-08 07:59:59.900   -0.083680
2019-05-08 08:00:00.000   -0.082227
Freq: 100L, Name: ventilation_5min_deriv, Length: 429000, dtype: float64

In [122]:
statistics = statistics.iloc[1:,0:]

In [14]:
statistics

,filename,ptype,BBI_mean,BBI_std,BBI_median,BBI_Q75,BBI_Q25,BB_RMSSD,BB_SDA,BB_ASD
0,icusleep_079_n05.h5,icu,3.069206,0.689685,3.1,3.4,2.7,0.738438,0.374757,0.513584


In [124]:
fig = plt.Figure(figsize = (7,7))
# g = sns.pairplot(, , height=1) # .sort_values('ptype')
g = sns.PairGrid(data= statistics.iloc[:,1:], hue="ptype" , height = 1)
g.map_upper(plt.scatter, s=6, alpha=0.7)
g.map_diag(plt.hist, bins=9, histtype='step')
g.map_lower(sns.kdeplot)
plt.savefig('p')

Canvas(toolbar=Toolbar(toolitems=[('Home', 'Reset original view', 'home', 'home'), ('Back', 'Back to previous …

In [70]:
fig = plt.Figure(figsize = (7,7))
# g = sns.pairplot(, , height=1) # .sort_values('ptype')
g = sns.PairGrid(data= statistics.iloc[:,1:], vars=['BBI_mean','BBI_std','BB_RMSSD','BB_SDA','BB_ASD'], hue="ptype" , height = 1.2)
g.map_upper(plt.scatter, s=6, alpha=0.7)
g.map_diag(plt.hist, bins=9, histtype='step')
g.map_lower(sns.kdeplot)
plt.savefig('BBI_variables_distribution_pairplot.png')

Canvas(toolbar=Toolbar(toolitems=[('Home', 'Reset original view', 'home', 'home'), ('Back', 'Back to previous …

In [71]:
print(f"Mean   Breath-Breath Interval: {data['BB_interval'][data.peaks==1].mean():.1f}")
print(f"Std.   Breath-Breath Interval: {data['BB_interval'][data.peaks==1].std():.1f}")
print(f"Median Breath-Breath Interval: {data['BB_interval'][data.peaks==1].median()}")
print(f"0.75Q    Breath-Breath Interval: {data['BB_interval'][data.peaks==1].quantile(0.75)}")
print(f"0.25Q    Breath-Breath Interval: {data['BB_interval'][data.peaks==1].quantile(0.25)}")
fig, ax = plt.subplots(1, figsize=(5,2))
ax.hist(data['BB_interval'][data.peaks==1], color='red', bins=range(0,30,1), histtype = 'step') # density=True


Mean   Breath-Breath Interval: 3.7
Std.   Breath-Breath Interval: 4.6
Median Breath-Breath Interval: 2.3
0.75Q    Breath-Breath Interval: 3.9
0.25Q    Breath-Breath Interval: 1.8


Canvas(toolbar=Toolbar(toolitems=[('Home', 'Reset original view', 'home', 'home'), ('Back', 'Back to previous …

(array([1.360e+02, 4.393e+03, 2.584e+03, 1.694e+03, 8.860e+02, 5.660e+02,
        4.080e+02, 2.250e+02, 1.670e+02, 1.260e+02, 8.500e+01, 7.500e+01,
        5.000e+01, 4.100e+01, 3.100e+01, 3.200e+01, 2.600e+01, 2.500e+01,
        1.500e+01, 1.200e+01, 8.000e+00, 1.000e+01, 1.400e+01, 1.000e+01,
        6.000e+00, 9.000e+00, 4.000e+00, 1.000e+01, 5.000e+00]),
 array([ 0,  1,  2,  3,  4,  5,  6,  7,  8,  9, 10, 11, 12, 13, 14, 15, 16,
        17, 18, 19, 20, 21, 22, 23, 24, 25, 26, 27, 28, 29]),
 <a list of 1 Patch objects>)

In [158]:
statistics_s = statistics[statistics.ptype=='sleeplab']
statistics_i = statistics[statistics.ptype=='icu']

In [159]:
statistics_i.head(2)
statistics_s.head(2)

,filename,ptype,comment,BBI_mean,BBI_std,BBI_median,BBI_Q75,BBI_Q25,BB_RMSSD,BB_SDA,BB_ASD
0,psg_airgo_10hz_055.h5,sleeplab,low AHI,4.390818,2.239450,3.9,4.2,3.7,2.632965,1.133255,1.688957
0,psg_airgo_10hz_424.h5,sleeplab,low AHI,4.547718,0.975419,4.6,4.9,4.3,1.129158,0.354469,0.754915


In [160]:
fig, ax = plt.subplots(figsize=(10,10))

ax.scatter(statistics_s['BBI_mean'], statistics_s['BBI_std'], c='blue', label='Sleeplab',alpha=0.7)
for i in range(statistics_s.shape[0]):
    ax.annotate(statistics_s['comment'].iloc[i] +','+ statistics_s['filename'].iloc[i][-6:-3], (statistics_s['BBI_mean'].iloc[i], statistics_s['BBI_std'].iloc[i]),  fontsize=7)

ax.scatter(statistics_i['BBI_mean'], statistics_i['BBI_std'], c='red', label='ICU',alpha=0.7)
for i in range(statistics_i.shape[0]):
    ax.annotate(statistics_i['filename'].iloc[i][-10:-3], (statistics_i['BBI_mean'].iloc[i], statistics_i['BBI_std'].iloc[i]), fontsize=7)

fig.suptitle('Breath-Breath-Interval Mean and STD', fontsize=9)
fig.legend(fontsize=9, loc=(0.15,0.76))
fig.text(0.02, 0.5, 'STD', ha='center', va='center', rotation='vertical')
fig.text(0.5, 0.02, 'Mean', ha='center', va='center')
plt.savefig('Breath-Breath_Mean_vs_Std.png')

Canvas(toolbar=Toolbar(toolitems=[('Home', 'Reset original view', 'home', 'home'), ('Back', 'Back to previous …

In [73]:
fig = plt.figure(figsize=(6,4))
# plt.title(')

n_sleeplab = 0
n_icu = 0

for i, filename in enumerate(dall.keys()):
    ax = fig.add_subplot(8,1,i+1)
    data = dall[filename]
    print(filename)
    print(f"Mean   Breath-Breath Interval: {data['BB_interval'][data.peaks==1].mean():.1f}")
    print(f"Std.   Breath-Breath Interval: {data['BB_interval'][data.peaks==1].std():.1f}")
    print(f"Median Breath-Breath Interval: {data['BB_interval'][data.peaks==1].median()}")
    print(f"0.75Q    Breath-Breath Interval: {data['BB_interval'][data.peaks==1].quantile(0.75)}")
    print(f"0.25Q    Breath-Breath Interval: {data['BB_interval'][data.peaks==1].quantile(0.25)}")
    
    if 'psg' in filename: 
        n_sleeplab += 1
        color = 'Blue'
        label = 'Sleeplab Night'
        if n_sleeplab > 1: label=None
    else: 
        n_icu += 1
        color = 'Red'
        label = 'ICU Night'
        if n_icu > 1: label=None
        
    ax.hist(data['BB_interval'][data.peaks==1], color=color, bins=np.arange(0,10,0.5), histtype = 'step', label=label) # density=True
    ax.set_xticks([0,1,2,3,4,5,6,7,8,9])
ax.set_xlabel('Time (seconds)')
fig.suptitle('Breath-Breath Intervals', fontsize=9)
fig.legend(fontsize=9)
fig.text(0.02, 0.5, 'Frequency', ha='center', va='center', rotation='vertical')
plt.savefig('BBI_Histograms.png')

Canvas(toolbar=Toolbar(toolitems=[('Home', 'Reset original view', 'home', 'home'), ('Back', 'Back to previous …

psg_airgo_10hz_055.h5
Mean   Breath-Breath Interval: 11.7
Std.   Breath-Breath Interval: 44.5
Median Breath-Breath Interval: 4.0
0.75Q    Breath-Breath Interval: 7.2
0.25Q    Breath-Breath Interval: 3.8
psg_airgo_10hz_424.h5
Mean   Breath-Breath Interval: 4.6
Std.   Breath-Breath Interval: 1.0
Median Breath-Breath Interval: 4.6
0.75Q    Breath-Breath Interval: 4.9
0.25Q    Breath-Breath Interval: 4.3
psg_airgo_10hz_332.h5
Mean   Breath-Breath Interval: 4.2
Std.   Breath-Breath Interval: 2.4
Median Breath-Breath Interval: 3.7
0.75Q    Breath-Breath Interval: 4.2
0.25Q    Breath-Breath Interval: 3.2
psg_airgo_10hz_317.h5
Mean   Breath-Breath Interval: 5.9
Std.   Breath-Breath Interval: 5.1
Median Breath-Breath Interval: 4.3
0.75Q    Breath-Breath Interval: 6.0
0.25Q    Breath-Breath Interval: 3.4
psg_airgo_10hz_044.h5
Mean   Breath-Breath Interval: 5.1
Std.   Breath-Breath Interval: 3.0
Median Breath-Breath Interval: 4.5
0.75Q    Breath-Breath Interval: 5.1
0.25Q    Breath-Breath Interva

ValueError: num must be 1 <= num <= 8, not 9

In [74]:
fig, ax = plt.subplots(figsize=(4,4))

n_sleeplab = 0
n_icu = 0

for i, filename in enumerate(dall.keys()):
    data = dall[filename]
   
    if 'psg' in filename: 
        n_sleeplab += 1
        color = 'Blue'
        label = 'Sleeplab Night'
        if n_sleeplab > 1: label=None
    else: 
        n_icu += 1
        color = 'Red'
        label = 'ICU Night'
        if n_icu > 1: label=None
            
#     print(f"Mean   Breath-Breath Interval: {data['BB_interval'][data.peaks==1].mean():.1f}")
#     print(f"Std.   Breath-Breath Interval: {data['BB_interval'][data.peaks==1].std():.1f}")
    ax.scatter(data['BB_interval'][data.peaks==1].mean(), data['BB_interval'][data.peaks==1].std(), c=color, label=label,alpha=0.7)

fig.suptitle('Breath-Breath Mean and STD', fontsize=9)
fig.legend(fontsize=9, loc=(0.15,0.76))
fig.text(0.02, 0.5, 'STD', ha='center', va='center', rotation='vertical')
fig.text(0.5, 0.02, 'Mean', ha='center', va='center')
plt.savefig('Breath-Breath_Mean_vs_Std.png')

Canvas(toolbar=Toolbar(toolitems=[('Home', 'Reset original view', 'home', 'home'), ('Back', 'Back to previous …

In [13]:
### RMS
fig, ax = plt.subplots(figsize=(4,4))

n_sleeplab = 0
n_icu = 0

for i, filename in enumerate(dall.keys()):
    data = dall[filename]
   
    if 'psg' in filename: 
        n_sleeplab += 1
        color = 'Blue'
        label = 'Sleeplab Night'
        if n_sleeplab > 1: label=None
    else: 
        n_icu += 1
        color = 'Red'
        label = 'ICU Night'
        if n_icu > 1: label=None
            
#     print(f"Mean   Breath-Breath Interval: {data['BB_interval'][data.peaks==1].mean():.1f}")
#     print(f"Std.   Breath-Breath Interval: {data['BB_interval'][data.peaks==1].std():.1f}")
    BB_mean = data['BB_interval'][data.peaks==1].mean()
    BB_std = data['BB_interval'][data.peaks==1].std()
    BB_rms = np.sqrt(np.mean(data['BB_interval'][data.peaks==1].diff().values[2:]**2))
    ax.scatter(BB_std, BB_rms, c=color, label=label,alpha=0.7)

fig.suptitle('Breath-Breath successive RMS and STD', fontsize=9)
fig.legend(fontsize=9, loc=(0.15,0.76))
fig.text(0.02, 0.5, 'STD', ha='center', va='center', rotation='vertical')
fig.text(0.5, 0.02, 'RMS SBB', ha='center', va='center')




Canvas(toolbar=Toolbar(toolitems=[('Home', 'Reset original view', 'home', 'home'), ('Back', 'Back to previous …

Text(0.5, 0.02, 'RMS SBB')

(432001,)

In [211]:
for i in range(30):
    plt.close()

In [53]:
data_sel = data.loc[pd.Timestamp('2019-01-25 05:05:00'):pd.Timestamp('2019-01-25 05:08:00')]

In [54]:
fig, (ax1, ax2, ax3, ax4) = plt.subplots(4,1, figsize=(10,3), sharex=True)
ax1.plot(data_sel.band, c='blue', lw=0.6)
ax1.scatter(data_sel[data_sel.peaks==1].index , data_sel[data_sel.peaks==1].band, c='red', s=5)

# ax2.plot(data_sel.rr_10s, c='green', lw=0.6)
ax2.plot(data_sel.rr_10s_smooth, c='blue', lw=0.6)
ax3.plot(data_sel.index, data_sel.ventilation_10s_smooth, c='blue', lw=0.6)
ax4.plot(data_sel.index, data_sel.BB_interval, c='green', lw=1)


Canvas(toolbar=Toolbar(toolitems=[('Home', 'Reset original view', 'home', 'home'), ('Back', 'Back to previous …

###  frequency spectrum analysis

In [5]:
from mne.time_frequency import psd_array_multitaper
from scipy.integrate import simps

In [6]:
def bandpower(data, sf, band, method='welch', window_sec=None, relative=False):
    """Compute the average power of the signal x in a specific frequency band.

    Requires MNE-Python >= 0.14.

    Parameters
    ----------
    data : 1d-array
      Input signal in the time-domain.
    sf : float
      Sampling frequency of the data.
    band : list
      Lower and upper frequencies of the band of interest.
    method : string
      Periodogram method: 'welch' or 'multitaper'
    window_sec : float
      Length of each window in seconds. Useful only if method == 'welch'.
      If None, window_sec = (1 / min(band)) * 2.
    relative : boolean
      If True, return the relative power (= divided by the total power of the signal).
      If False (default), return the absolute power.

    Return
    ------
    bp : float
      Absolute or relative band power.
    """
    from scipy.signal import welch
    from scipy.integrate import simps
    from mne.time_frequency import psd_array_multitaper

    band = np.asarray(band)
    low, high = band

    # Compute the modified periodogram (Welch)
    if method == 'welch':
        if window_sec is not None:
            nperseg = window_sec * sf
        else:
            nperseg = (2 / low) * sf

        freqs, psd = welch(data, sf, nperseg=nperseg)

    elif method == 'multitaper':
        psd, freqs = psd_array_multitaper(data, sf, adaptive=True,
                                          normalization='full', verbose=0)

    # Frequency resolution
    freq_res = freqs[1] - freqs[0]

    # Find index of band in frequency vector
    idx_band = np.logical_and(freqs >= low, freqs <= high)

    # Integral approximation of the spectrum using parabola (Simpson's rule)
    bp = simps(psd[idx_band], dx=freq_res)

    if relative:
        bp /= simps(psd, dx=freq_res)
    return bp

In [233]:
dall.keys()

dict_keys(['psg_airgo_10hz_001.h5', 'psg_airgo_10hz_002.h5', 'psg_airgo_10hz_003.h5', 'psg_airgo_10hz_004.h5', 'psg_airgo_10hz_005.h5', 'icusleep_021_n02.h5', 'icusleep_025_n04.h5', 'icusleep_079_n05.h5'])

array([ nan,  nan,  nan, ..., 685., 682.,  nan], dtype=float16)

['psg_airgo_10hz_001.h5',
 'psg_airgo_10hz_002.h5',
 'psg_airgo_10hz_003.h5',
 'psg_airgo_10hz_004.h5',
 'psg_airgo_10hz_005.h5']

In [254]:
# d = data.band.values
filename=sleeplab_files[2]
d = dall[filename].band.dropna().values
d.shape

band = [0, 0.05]
bp_abs = bandpower(d[:30*fs], fs, band, method='multitaper', window_sec=None, relative=False)
print(bp_abs)
bp_rel = bandpower(d[:30*fs], fs, band, method='multitaper', window_sec=None, relative=True)
print(bp_rel)

150.2570322807783
0.11189871173180437


In [255]:
dB = True

psd, freqs = psd_array_multitaper(d[:30], fs, adaptive=True, normalization='full', verbose=0)

if dB: psd = 10 * np.log10(psd)

sc = 'slategrey'
fig, ax = plt.subplots(figsize=(5,3))
ax.stem(freqs, psd, linefmt=sc, basefmt=" ", markerfmt=" ", use_line_collection=True)
lc, lw = 'k', 2
ax.plot(freqs, psd, lw=lw, color=lc)

ax.set_xlabel('Frequency (Hz)')
if not dB:
    ax.set_ylabel('Power spectral density (amp^2/Hz)')
else:
    ax.set_ylabel('Decibels (dB / Hz)')
plt.tight_layout()

Canvas(toolbar=Toolbar(toolitems=[('Home', 'Reset original view', 'home', 'home'), ('Back', 'Back to previous …

### higher Freq Res, les time res:

In [256]:
# window length
wl = 2/0.01
print(f'window length in seconds: {wl}')
stepsize = 5 # seconds

window length in seconds: 200.0


In [257]:
import time
start = time.time()
psds = [10 * np.log10(psd_array_multitaper(d[int(x):int(x+wl*fs)], fs, adaptive=True, normalization='full', verbose=0)[0][:,np.newaxis]) for x in np.arange(0,len(d)-wl*fs,stepsize*fs)]
x=0
freqs = psd_array_multitaper(d[int(x):int(x+wl*fs)], fs, adaptive=True, normalization='full', verbose=0)[1] 
print(time.time()-start)

91.21450638771057


###  lower Freq Res, higher time res:

In [258]:
wl = 2/0.02
print(f'window length in seconds: {wl}')
stepsize = 5 # seconds

window length in seconds: 100.0


In [259]:
import time
# window length

start = time.time()
psds2 = [10 * np.log10(psd_array_multitaper(d[int(x):int(x+wl*fs)], fs, adaptive=True, normalization='full', verbose=0)[0][:,np.newaxis]) for x in np.arange(0,len(d)-wl*fs,stepsize*fs)]
x=0
freqs2 = psd_array_multitaper(d[int(x):int(x+wl*fs)], fs, adaptive=True, normalization='full', verbose=0)[1] 
print(time.time()-start)




47.76581954956055


In [260]:
spectrogram = np.concatenate(psds, axis=1)
spectrogram2 = np.concatenate(psds2, axis=1)

In [261]:
freq_resolution = freqs[1] - freqs[0]
idx_1hz = np.where(freqs==1)[0][0]

freq_resolution2 = freqs2[1] - freqs2[0]
print(freq_resolution2)
idx_1hz2 = np.where(freqs2==1)[0][0]

0.01


In [ ]:
# fig, ax = plt.subplots()
# ax.imshow(spectrogram, origin='lower', aspect='auto')
# ax.set_yticks(np.arange(0, len(freqs), 1/freq_resolution))
# ax.set_yticklabels(freqs[::int(1/freq_resolution)])
# plt.show()

In [262]:
fig= plt.figure(figsize=(12,6))
ax1 = fig.add_subplot(3,1,1)
ax1.plot(d,c='k',lw=0.5)
ax1.set_xlim([0, len(d)])
ax2 = fig.add_subplot(3,1,2, sharex = ax1)
ax2.imshow(spectrogram[:idx_1hz,:], origin='lower', aspect='auto', cmap=cm.jet, vmin=np.quantile(spectrogram[:idx_1hz,:],0.05),extent=[0, len(d), 0, freqs[idx_1hz]], vmax=np.quantile(spectrogram[:idx_1hz,:],0.995))
ax3 = fig.add_subplot(3,1,3, sharex = ax1)
ax3.imshow(spectrogram2[:idx_1hz2,:], origin='lower', aspect='auto', cmap=cm.jet, vmin=np.quantile(spectrogram2[:idx_1hz2,:],0.05),extent=[0, len(d), 0, freqs2[idx_1hz2]], vmax=np.quantile(spectrogram2[:idx_1hz2,:],0.995))
plt.show()
fig.suptitle(filename)

Canvas(toolbar=Toolbar(toolitems=[('Home', 'Reset original view', 'home', 'home'), ('Back', 'Back to previous …

Text(0.5, 0.98, 'psg_airgo_10hz_003.h5')

In [253]:
fig= plt.figure(figsize=(12,6))
ax1 = fig.add_subplot(3,1,1)
ax1.plot(d,c='k',lw=0.5)
ax1.set_xlim([0, len(d)])
ax2 = fig.add_subplot(3,1,2, sharex = ax1)
ax2.imshow(spectrogram[:idx_1hz,:], origin='lower', aspect='auto', cmap=cm.jet, vmin=np.quantile(spectrogram[:idx_1hz,:],0.05),extent=[0, len(d), 0, freqs[idx_1hz]], vmax=np.quantile(spectrogram[:idx_1hz,:],0.995))
ax3 = fig.add_subplot(3,1,3, sharex = ax1)
ax3.imshow(spectrogram2[:idx_1hz2,:], origin='lower', aspect='auto', cmap=cm.jet, vmin=np.quantile(spectrogram2[:idx_1hz2,:],0.05),extent=[0, len(d), 0, freqs2[idx_1hz2]], vmax=np.quantile(spectrogram2[:idx_1hz2,:],0.995))
plt.show()
fig.suptitle(filename)

Canvas(toolbar=Toolbar(toolitems=[('Home', 'Reset original view', 'home', 'home'), ('Back', 'Back to previous …

Text(0.5, 0.98, 'psg_airgo_10hz_001.h5')

In [230]:
fig= plt.figure(figsize=(12,6))
ax1 = fig.add_subplot(3,1,1)
ax1.plot(d,c='k',lw=0.5)
ax1.set_xlim([0, len(d)])
ax2 = fig.add_subplot(3,1,2, sharex = ax1)
ax2.imshow(spectrogram[:idx_1hz,:], origin='lower', aspect='auto', cmap=cm.jet, vmin=np.quantile(spectrogram[:idx_1hz,:],0.05),extent=[0, len(d), 0, freqs[idx_1hz]], vmax=np.quantile(spectrogram[:idx_1hz,:],0.995))
ax3 = fig.add_subplot(3,1,3, sharex = ax1)
ax3.imshow(spectrogram2[:idx_1hz2,:], origin='lower', aspect='auto', cmap=cm.jet, vmin=np.quantile(spectrogram2[:idx_1hz2,:],0.05),extent=[0, len(d), 0, freqs2[idx_1hz2]], vmax=np.quantile(spectrogram2[:idx_1hz2,:],0.995))
plt.show()

Canvas(toolbar=Toolbar(toolitems=[('Home', 'Reset original view', 'home', 'home'), ('Back', 'Back to previous …

In [126]:
freqs[0:idx_1hz:int(1/freq_resolution/10)]

array([0. , 0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9])

In [222]:
for i in range(30):
    plt.close()

In [106]:
freqs.shape

(1001,)

In [108]:
spectrogram.shape

(1001, 140)

In [171]:
spectrogram[:idx_1hz,:].shape

(200, 8601)

In [172]:
np.repeat(spectrogram[:idx_1hz,:],stepsize*fs,axis=1).shape

(200, 430050)